**#Step 2. MetaFX и MASH#**

**Substep 2.1** MetaFX counting k-mers (script run_metafx_step1.sbatch)

Creates symbolic links with names sample_r1.fastq.gz / sample_r2.fastq.gz (lowercase). 
Generates sample_categories.txt in the 'sample'\t'category' format (without header)
Runs metafx unique without --kmers-dir to count kmers.

As a result of the script running in the metafx_output directory:
feature_table.tsv: this is a sample-feature table in a format ready for machine learning.

In [ ]:
#SBATCH -J metafx_step1
#SBATCH --output=/mnt/tank/scratch/ris/SRR_files/Logs/metafx_step1_%j.out
#SBATCH --error=/mnt/tank/scratch/ris/SRR_files/Logs/metafx_step1_%j.err
#SBATCH -n 1
#SBATCH --cpus-per-task=16
#SBATCH --mem=128G
#SBATCH -t 24:00:00

set -e
set -u

cd /mnt/tank/scratch/ris/SRR_files || exit 1

source /nfs/home/ris/miniforge3/etc/profile.d/mamba.sh
mamba activate snakemake

# Creating links sample_r1.fastq.gz / sample_r2.fastq.gz
for r1 in *_1_trimmed_paired.fastq.gz; do
    sample_orig=$(basename "$r1" _1_trimmed_paired.fastq.gz)
    sample_lower=$(echo "$sample_orig" | tr '[:upper:]' '[:lower:]')
    r2="${sample_orig}_2_trimmed_paired.fastq.gz"
    if [ -f "$r2" ]; then
        ln -sf "$r1" "${sample_lower}_r1.fastq.gz"
        ln -sf "$r2" "${sample_lower}_r2.fastq.gz"
    else
        echo "Warning: $r2 not found for $sample_orig"
    fi
done

# Creation sample_categories.txt (approximate category)
python3 - <<EOF # the heredoc operator. It tells the shell that everything up to the EOF line is passed as standard input to the python3 command.
import pandas as pd
import os

metadata = pd.read_csv("SraRunTable.csv")
metadata.rename(columns={'Run': 'sample'}, inplace=True)
metadata['group'] = metadata['Fracture'].apply(lambda x: 'healthy' if x == 0 else 'case')

with open("sample_categories.txt", "w") as f:
    for _, row in metadata.iterrows():
        sample_lower = row['sample'].lower()
        group = row['group']
        # Check for links
        r1 = f"{sample_lower}_r1.fastq.gz"
        r2 = f"{sample_lower}_r2.fastq.gz"
        if os.path.exists(r1) and os.path.exists(r2):
            f.write(f"{sample_lower}\t{group}\n")
        else:
            print(f"Warning: {sample_lower} missing reads")
EOF

# 3. Launch metafx unique (without --kmers-dir) – it will calculate kmers by itself
metafx unique -t 16 -m 2G -w wd_unique -k 31 -i sample_categories.txt

**Substep 2.1.1** 

The files srr*_r1.kmers will appear in the folder wd_unique/kmers/kmers/.bin and srr*_r2.kmers.bin.
after step 1, it is needed to manually fix sample_categories.txt 

Fix it sample_categories.txt - remove suffixes _r1.fastq.gz , leaving only the IDs:

In [ ]:
sed -i 's/_r1\.fastq\.gz//' sample_categories.txt

# Create symbolic links to kmers without the _r1 suffix (so that the program can find them):

cd wd_unique/kmers/kmers
for f in *_r1.kmers.bin; do
    base=$(basename "$f" _r1.kmers.bin)
    ln -s "$f" "${base}.kmers.bin"
done
cd /mnt/tank/scratch/ris/SRR_files

# Delete the old temporary folders of the second stage to avoid the issue of overwriting, 
# which cannot be accepted or rejected interactively during the script:

rm -rf wd_unique/unique_kmers_healthy wd_unique/unique_kmers_case
rm -rf wd_unique/components_healthy wd_unique/components_case
rm -rf wd_unique/contigs_healthy wd_unique/contigs_case

**Substep 2.1.2** Extracting features using ready-made kmers (run_metafx_step2.sbatch script)

In [ ]:
#!/bin/bash
#SBATCH -J metafx_step2
#SBATCH --output=/mnt/tank/scratch/ris/SRR_files/Logs/metafx_step2_%j.out
#SBATCH --error=/mnt/tank/scratch/ris/SRR_files/Logs/metafx_step2_%j.err
#SBATCH -n 1
#SBATCH --cpus-per-task=16
#SBATCH --mem=128G
#SBATCH -t 12:00:00

set -e
set -u

cd /mnt/tank/scratch/ris/SRR_files || exit 1

source /nfs/home/ris/miniforge3/etc/profile.d/mamba.sh
mamba activate snakemake

# Deleting the old temporary folders of step 2
rm -rf wd_unique/unique_kmers_healthy wd_unique/unique_kmers_case
rm -rf wd_unique/components_healthy wd_unique/components_case
rm -rf wd_unique/contigs_healthy wd_unique/contigs_case

# Launching metafx unique with --kmers-dir
metafx unique -t 16 -m 2G -w wd_unique -k 31 -i sample_categories.txt --kmers-dir wd_unique/kmers/kmers

# After that, the final files will appear in wd_unique/: feature_table.tsv

After the successful completion of the MetaFX pipeline, the following were received in the working directory *wd_unique/*:

*feature_table.tsv* – table of features (rows – features, columns – samples)

*samples_categories.tsv* – matching samples to categories (healthy / case)

*categories_samples.tsv* – grouping samples by category

Subfolders *contigs_health/* and *contigs_case/* with FASTA files of contigs for each group

**Substep 2.1.3** Clearing the feature table (removing duplicate columns)

In [ ]:
cd /mnt/tank/scratch/ris/SRR_files
cp wd_unique/feature_table.tsv wd_unique/feature_table_original.tsv

Python script for cleaning:

In [ ]:
import pandas as pd

feat = pd.read_csv('wd_unique/feature_table.tsv', sep='\t', index_col=0)
# Remove the suffix _r1 from the column names
feat.columns = [col.replace('_r1', '') for col in feat.columns]
# Removing duplicates (leaving the first column for each sample)
feat = feat.loc[:, ~feat.columns.duplicated()]
feat.to_csv('wd_unique/feature_table.tsv', sep='\t')
print(f"There were columns: {pd.read_csv('wd_unique/feature_table_original.tsv', sep='\t', index_col=0).shape[1]}")
print(f"Become columns{feat.shape[1]}")

**Substep 2.1.4**  Post-processing and analysis of MetaFX results for BMD and Fracture

Visualization of PCA. Built-in PCA MetaFX 

SLURM script run_pca.sbatch:

In [ ]:
#!/bin/bash
#SBATCH -J pca
#SBATCH --output=/mnt/tank/scratch/ris/SRR_files/Logs/pca_%j.out
#SBATCH --error=/mnt/tank/scratch/ris/SRR_files/Logs/pca_%j.err
#SBATCH -n 1
#SBATCH --cpus-per-task=4
#SBATCH --mem=16G
#SBATCH -t 1:00:00

cd /mnt/tank/scratch/ris/SRR_files # || exit 1 if the transition fails, terminate the script with error code 1. This prevents further execution of commands in the wrong place.
source /nfs/home/ris/miniforge3/etc/profile.d/mamba.sh
mamba activate snakemake

metafx pca -w wd_pca -f wd_unique/feature_table.tsv -i wd_unique/samples_categories.tsv --show

# PCA for bone density (but everything is done in approximately the same way)
(snakemake) ris@sphinx:/mnt/tank/scratch/ris/SRR_files/wd_unique_bmd$ metafx pca \
    -w wd_pca_bmd \
    -f feature_table.tsv \
    -i samples_categories.tsv \
    --show
metafx pca -w wd_pca_bmd -f feature_table.tsv -i samples_categories.tsv --show

**Substep 2.1.5** Random Forest training with cross-validation
SLURM script run_cv.sbatch:

In [ ]:
#!/bin/bash
#SBATCH -J rf_cv
#SBATCH --output=/mnt/tank/scratch/ris/SRR_files/Logs/rf_cv_%j.out
#SBATCH --error=/mnt/tank/scratch/ris/SRR_files/Logs/rf_cv_%j.err
#SBATCH -n 1
#SBATCH --cpus-per-task=16
#SBATCH --mem=64G
#SBATCH -t 4:00:00

cd /mnt/tank/scratch/ris/SRR_files || exit 1
source /nfs/home/ris/miniforge3/etc/profile.d/mamba.sh
mamba activate snakemake

metafx cv -t 16 -w wd_cv -f wd_unique/feature_table.tsv -i wd_unique/samples_categories.tsv -n 5 --grid

Clearing feature tables (removing duplicate columns)

Problem: feature_table.tsv contained duplicate columns for each sample (for example, srr... and srr..._r1), which prevented further analysis.

In [ ]:
# Commands for BMD (similar for Fracture):

cd /mnt/tank/scratch/ris/SRR_files/wd_unique_bmd
cp feature_table.tsv feature_table_original.tsv
sed -i '2d' feature_table.tsv   # removing the duplicate header line

Random Forest with an assessment of the importance of features (Python, running through SLURM)

we train a classifier to separate groups (normal/low and healthy/case) and determine the most informative features.

Script used train_rf.py (universal):

In [ ]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.metrics import classification_report, confusion_matrix
import joblib
import matplotlib.pyplot as plt

# Uploading data
X = pd.read_csv('feature_table.tsv', sep='\t', index_col=0).T.fillna(0)
y_df = pd.read_csv('samples_categories.tsv', sep='\t', header=None, names=['sample', 'group'])
X.index = X.index.str.lower()
y_df['sample'] = y_df['sample'].str.lower()
common = X.index.intersection(y_df['sample'])
X = X.loc[common]
y = y_df.set_index('sample').loc[common, 'group']

# Separation and training
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=42, stratify=y)
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

# Rating
y_pred = rf.predict(X_test)
print(classification_report(y_test, y_pred))

# Importance of signs
importances = rf.feature_importances_
imp_df = pd.DataFrame({'feature': X.columns, 'importance': importances}).sort_values('importance', ascending=False)
imp_df.to_csv('feature_importance.csv', index=False)

# Chart of the top 20
top = imp_df.head(20)
plt.figure(figsize=(10,8))
plt.barh(top['feature'], top['importance'], color='skyblue')
plt.savefig('top20_features.png', dpi=300)

SLURM script run_rf.sbatch:

In [ ]:
#!/bin/bash
#SBATCH -J rf_bmd
#SBATCH --output=Logs/rf_%j.out
#SBATCH --error=Logs/rf_%j.err
#SBATCH -n 1 --cpus-per-task=8 --mem=32G -t 1:00:00
cd /mnt/tank/scratch/ris/SRR_files/wd_unique_bmd
source /nfs/home/ris/miniforge3/etc/profile.d/mamba.sh
mamba activate snakemake
python3 train_rf.py

Sequence extraction for the top 20 important features

get FASTA files with nucleotide sequences corresponding to the most important components of MetaFX for subsequent annotation (BLAST)

Script extract_top_by_index.py (for Fracture, it is the same for BMD):

In [ ]:
import pandas as pd
from Bio import SeqIO

imp = pd.read_csv('rf_results/feature_importance.csv')
top20 = imp.head(20)['feature'].tolist()
with open('top20_features.fasta', 'w') as out:
    for feat in top20:
        group, idx = feat.split('_')
        idx = int(idx)
        fasta_file = f'contigs_{group}/components.seq.fasta'
        records = list(SeqIO.parse(fasta_file, 'fasta'))
        if idx < len(records):
            rec = records[idx]
            out.write(f'>{feat} ({rec.id})\n{rec.seq}\n')

In [ ]:
python3 extract_top_by_index.py

# Result: top20_features.fasta file (for Fracture) and similarly for BMD (with changed group names) 
#(aviable in linked zenodo repo and later are needed for Marker interpritation step)

**Subset 2.2.1** Mash

We run Mash in parallel with (or after) MetaFX. Script *run_mash.sbatch*:

Mash uses the MinHash algorithm to quickly estimate the similarity between all samples based on their k-mer composition.

*mash_output/distances.tab*: The main result of Mash. This is a file with a matrix of pairwise distances between all samples.

The values (from 0 to 1) indicate the genetic distance between the samples. The closer the value is to 0, the more similar the two communities are. This matrix is the basis for building trees, clustering, and visualization.

*mash_output/all_sketches.msh*: Archive file with compressed signatures (sketch) of all samples. It can be used for quick comparison with new data without re-creating sketches.

In [ ]:
#!/bin/bash
#SBATCH -J mash
#SBATCH --output=/mnt/tank/scratch/ris/SRR_files/Logs/mash_%j.out
#SBATCH --error=/mnt/tank/scratch/ris/SRR_files/Logs/mash_%j.err
#SBATCH -n 1
#SBATCH --cpus-per-task=8
#SBATCH --mem=32G
#SBATCH -t 24:00:00

set -e
set -u

cd /mnt/tank/scratch/ris/SRR_files || exit 1

source /nfs/home/ris/miniforge3/etc/profile.d/mamba.sh
mamba activate snakemake

echo "Mash started at $(date)"

MASH_WD="mash_output"
mkdir -p "$MASH_WD"

# Create a sketch for each sample by concatenating R1 and R2
for r1 in *_1_trimmed_paired.fastq.gz; do
    sample_orig=$(basename "$r1" _1_trimmed_paired.fastq.gz)
    r2="${sample_orig}_2_trimmed_paired.fastq.gz"
    if [ -f "$r2" ]; then
        cat "$r1" "$r2" | mash sketch -m 2 -o "$MASH_WD/${sample_orig}" -
    fi
done

# Combine all the sketches into one file
mash sketch -o "$MASH_WD/all_sketches" "$MASH_WD"/*.msh

# Building a distance matrix
mash dist "$MASH_WD/all_sketches.msh" "$MASH_WD/all_sketches.msh" > "$MASH_WD/distances.tab"
echo " Mash finished at $(date)"

**Substep 2.2.2** Hierarchical clustering and dendrogram visualization of beta diversity distances with sample annotations (fracture status and BMD category)

Hierarchical clustering is performed using the UPGMA (average linkage) method. The resulting dendrogram is annotated with two types of metadata:(1) fracture status (presence/absence marked with an asterisk) obtained from the SraRunTable, and (2) bone mineral density category (normal/low) provided by list (according to Step0, substep 0.6). Leaves are colored according to the BMD category (green = normal, red = low)

In [ ]:
import pandas as pd
import numpy as np
from scipy.cluster.hierarchy import linkage, dendrogram
from scipy.spatial.distance import squareform
import matplotlib.pyplot as plt

samples = []
dist_dict = {}

with open('distances.tab', 'r') as f:
    for line in f:
        parts = line.strip().split('\t')
        if len(parts) < 3:
            continue
        s1, s2 = parts[0], parts[1]
        if s1 == s2:
            continue
        try:
            dist = float(parts[2])
        except:
            continue
        dist_dict[(s1, s2)] = dist
        dist_dict[(s2, s1)] = dist
        samples.extend([s1, s2])

unique_samples = sorted(set(samples))
n = len(unique_samples)
print(f"Found {n} unique samples.")

# Build distance matrix
dist_matrix = np.ones((n, n))
for i, s1 in enumerate(unique_samples):
    dist_matrix[i, i] = 0.0
    for j, s2 in enumerate(unique_samples):
        if i == j:
            continue
        if (s1, s2) in dist_dict:
            dist_matrix[i, j] = dist_dict[(s1, s2)]
        else:
            dist_matrix[i, j] = 1.0

dist_matrix = (dist_matrix + dist_matrix.T) / 2
condensed = squareform(dist_matrix, checks=False)

# 2.1 Fracture status from SraRunTable.csv
meta = pd.read_csv('SraRunTable.csv')
meta['has_fracture'] = meta['Fracture'] > 0   # True if fracture present

# Dictionary: sample -> fracture (bool)
fracture_dict = {}
for _, row in meta.iterrows():
    fracture_dict[row['Run']] = row['has_fracture']

# normal/low categories (BMD) (according to Step0, substep 0.6; From .txt file, but we can also make it more adorable)))
BMD_cat = """
SRR25006867	normal
SRR25006868	normal
SRR25006869	low
SRR25006870	low
SRR25006871	low
SRR25006872     low
SRR25006873	normal
SRR25006874	normal
SRR25006875	low
SRR25006876	normal
SRR25006877	normal
SRR25006878	normal
SRR25006879	low
SRR25006880	normal
SRR25006881	normal
SRR25006882	normal
SRR25006883	normal
SRR25006884	low
SRR25006885	normal
SRR25006886	normal
SRR25006887	low
SRR25006888	normal
SRR25006889	normal
SRR25006890	normal
SRR25006891	low
SRR25006892	low
SRR25006893	low
SRR25006894	normal
SRR25006895	low
SRR25006896	normal
SRR25006897	normal
SRR25006898	normal
SRR25006899	normal
SRR25006900	low
SRR25006901	normal
SRR25006902	normal
SRR25006903	normal
SRR25006904	normal
SRR25006905     low
SRR25006906	normal
SRR25006907	normal
SRR25006908	normal
SRR25006909	low
SRR25006910     normal
SRR25006911	normal
SRR25006912	normal
SRR25006913	normal
SRR25006914	low
SRR25006915	normal
SRR25006916	low
SRR25006917	low
SRR25006918	normal
SRR25006919	normal
SRR25006920	normal
SRR25006921	low
SRR25006922	low
SRR25006923	normal
SRR25006924	normal
SRR25006925	low
"""

category_dict = {}
for line in BMD_cat.strip().split('\n'):
    if not line.strip():
        continue
    parts = line.split()
    if len(parts) >= 2:
        samp = parts[0].strip().upper()
        cat = parts[1].strip().lower()
        category_dict[samp] = cat

# For each sample, create label: "ID" + "*" if fracture present
labels = []
leaf_colors = []
for s in unique_samples:
    # Basic label
    has_frac = fracture_dict.get(s, False)
    label = s
    if has_frac:
        label += '*'
    labels.append(label)
    
    # Color by normal/low category
    cat = category_dict.get(s)
    if cat == 'normal':
        leaf_colors.append('green')
    elif cat == 'low':
        leaf_colors.append('red')
    else:
        leaf_colors.append('gray')        

linkage_matrix = linkage(condensed, method='average')   # UPGMA

plt.figure(figsize=(16, 10))
dendrogram(linkage_matrix,
           labels=labels,
           leaf_rotation=90,
           leaf_font_size=8,
           color_threshold=0.5,
           above_threshold_color='gray')

# Change leaf colors according to category
ax = plt.gca()
for lbl, color in zip(ax.get_xmajorticklabels(), leaf_colors):
    lbl.set_color(color)

# Add legend
from matplotlib.patches import Patch
legend_elements = [Patch(facecolor='green', label='normal'),
                   Patch(facecolor='red', label='low'),
                   Patch(facecolor='none', label='* = fracture')]
ax.legend(handles=legend_elements, loc='upper right', fontsize=10)

plt.title('Dendrogram: green=normal, red=low, * = fracture')
plt.xlabel('Sample ID')
plt.ylabel('Distance')
plt.tight_layout()
plt.savefig('dendrogram_combined.png', dpi=150)
plt.show()

![Mash dendrogram](./images/BMD/dendrogram_combined.png)